**DoRothEA / CollecTRI** is a choice made for gene regulatory networks due to directionality property

**CollecTRI** is the better choice — it's a newer, more comprehensive resource **(~1,186 TFs vs DoRothEA's ~470-1,396 depending on confidence level)** built by aggregating signed TF-target interactions from 12+ resources, and benchmarking work (Müller-Dott et al. 2023)

In [1]:

from gears import PertData  # use the GEARS library to load the data and create the train/test split
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# split info: {'train_loader': 49849, 'val_loader': 10754, 'test_loader': 28754}
# === Condition-level counts (unique perturbations) ===
# train :  139 conditions  (99 single-gene, 39 double-gene)
# val   :   31 conditions  (13 single-gene, 18 double-gene)
# test  :  107 conditions  (36 single-gene, 71 double-gene)

# === Cell-level counts ===
# train_loader:  49849 cells
# val_loader  :  10754 cells
# test_loader :  28754 cells

SEED = 1
DATA_ROOT = "./data"
SPLIT_TYPE = "simulation"  # training set - single gene modification and some double gene modifications,
                            # testing set - tested on double-gene combinations where both genes were seen independently
                            # individually during training, but the combination itself is entirely new
save_dir = f"./data/my_norman_split_meta/{SPLIT_TYPE}_seed{SEED}"
os.makedirs(save_dir, exist_ok=True)


def load_pert_data(data_root):
    pert_data = PertData(data_root)
    pert_data.load(data_name="norman")
    pert_data.prepare_split(split=SPLIT_TYPE, seed=SEED)
    return pert_data


def export_split_conditions(pert_data):
    set2conditions = pert_data.set2conditions  # {'train': [...], 'val': [...], 'test': [...]}
    split_info = {
        "seed": SEED,
        "split_type": SPLIT_TYPE,
        "train_conditions": set2conditions["train"],
        "val_conditions": set2conditions.get("val", []),
        "test_conditions": set2conditions["test"],
    }
    with open(os.path.join(save_dir, "split_conditions.json"), "w") as f:
        json.dump(split_info, f, indent=2)
    return split_info


#Fetch the baseline expression values for the control cells, grouped by batch 
def compute_ctrl_baseline(adata, group_col="batch"):
    """Group-aware control baseline instead of one pooled global mean.
    Returns dict {group_key: np.array[n_genes]}, plus always includes 'global'."""
    ctrl_mask = adata.obs['condition'] == 'ctrl'
    baselines = {}

    expr_all = adata[ctrl_mask].X
    baselines["global"] = np.asarray(
        (expr_all.mean(axis=0) if hasattr(expr_all, "mean") else expr_all.mean(0))
    ).flatten()

    if group_col in adata.obs.columns:
        for group_id, sub in adata.obs.loc[ctrl_mask].groupby(group_col):
            idx = sub.index
            expr = adata[idx].X
            baselines[str(group_id)] = np.asarray(expr.mean(axis=0)).flatten()

    return baselines


class PerturbationDataset(Dataset):
    """
    Each sample:
        pert_genes  -> list[str], the perturbed gene name(s) for this cell (empty for ctrl)
        pert_idx    -> list[int], indices of those genes into gene_names (for embedding lookup)
        expression  -> torch.FloatTensor [n_genes], observed post-perturbation expression
        baseline    -> torch.FloatTensor [n_genes], matched control baseline for this cell's group
    """

    def __init__(self, adata, conditions, baseline_dict, gene_names, group_col="batch"):
        mask = adata.obs['condition'].isin(conditions)
        self.adata_sub = adata[mask]

        X = self.adata_sub.X
        self.expr = X.toarray() if hasattr(X, "toarray") else np.asarray(X)

        self.conditions = self.adata_sub.obs['condition'].values
        self.group_col = group_col if group_col in adata.obs.columns else None
        self.groups = self.adata_sub.obs[group_col].values if self.group_col else None

        self.gene_names = gene_names
        self.gene2idx = {g: i for i, g in enumerate(gene_names)}
        self.baseline_dict = baseline_dict  # {group_key: np.array[n_genes]}, includes "global"

    def __len__(self):
        return self.expr.shape[0]

    def __getitem__(self, idx):
        cond = self.conditions[idx]
        pert_genes = [g for g in cond.split('+') if g != 'ctrl']

        baseline_key = str(self.groups[idx]) if self.groups is not None else "global"
        baseline = self.baseline_dict.get(baseline_key, self.baseline_dict["global"])

        return {
            "pert_genes": pert_genes,  # e.g. ["KLF1", "MAP2K6"] or [] for ctrl
            "pert_idx": [self.gene2idx[g] for g in pert_genes if g in self.gene2idx],
            "expression": torch.tensor(self.expr[idx], dtype=torch.float32),
            "baseline": torch.tensor(baseline, dtype=torch.float32),
        }


def perturbation_collate(batch, pad_value=-1):
    """Pads variable-length pert_idx lists (0, 1, or 2+ perturbed genes) within a batch."""
    max_perts = max((len(b["pert_idx"]) for b in batch), default=0)
    max_perts = max(max_perts, 1)  # avoid zero-width tensor if a whole batch is ctrl-only
    pert_idx_padded = torch.full((len(batch), max_perts), pad_value, dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["pert_idx"])
        if n > 0:
            pert_idx_padded[i, :n] = torch.tensor(b["pert_idx"], dtype=torch.long)

    return {
        "pert_genes": [b["pert_genes"] for b in batch],  # list of lists, variable length (debug/logging)
        "pert_idx": pert_idx_padded,                       # [batch, max_perts], padded with pad_value
        "expression": torch.stack([b["expression"] for b in batch]),
        "baseline": torch.stack([b["baseline"] for b in batch]),
    }


def build_custom_datasets(pert_data, baseline_dict, split_info, group_col="batch"):
    gene_names = pert_data.adata.var_names.tolist()
    train_ds = PerturbationDataset(pert_data.adata, split_info["train_conditions"], baseline_dict, gene_names, group_col)
    val_ds = PerturbationDataset(pert_data.adata, split_info["val_conditions"], baseline_dict, gene_names, group_col)
    test_ds = PerturbationDataset(pert_data.adata, split_info["test_conditions"], baseline_dict, gene_names, group_col)
    return train_ds, val_ds, test_ds


def build_custom_loaders(train_ds, val_ds, test_ds, batch_size=32, test_batch_size=128):
    return {
        "train_loader": DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=perturbation_collate),
        "val_loader": DataLoader(val_ds, batch_size=test_batch_size, shuffle=False, collate_fn=perturbation_collate),
        "test_loader": DataLoader(test_ds, batch_size=test_batch_size, shuffle=False, collate_fn=perturbation_collate),
    }

def runner():
    pert_data = load_pert_data(DATA_ROOT)
    adata = pert_data.adata

    split_info = export_split_conditions(pert_data)
    baseline_dict = compute_ctrl_baseline(adata, group_col="batch")

    # Save metadata + baselines for reproducibility / downstream analysis
    baseline_keys = list(baseline_dict.keys())
    baseline_matrix = np.stack([baseline_dict[k] for k in baseline_keys])
    np.savez_compressed(
        os.path.join(save_dir, f"meta_seed{SEED}.npz"),
        gene_names=np.array(adata.var_names.tolist()),
        train_conditions=np.array(split_info["train_conditions"], dtype=object),
        val_conditions=np.array(split_info["val_conditions"], dtype=object),
        test_conditions=np.array(split_info["test_conditions"], dtype=object),
        baseline_keys=np.array(baseline_keys),
        baseline_matrix=baseline_matrix,
    )

    train_ds, val_ds, test_ds = build_custom_datasets(pert_data, baseline_dict, split_info, group_col="batch")
    loaders = build_custom_loaders(train_ds, val_ds, test_ds)
    return pert_data, loaders


def print_split_summary(pert_data, loaders):
    set2conditions = pert_data.set2conditions

    print("=== Condition-level counts (unique perturbations) ===")
    for split in ["train", "val", "test"]:
        n_conditions = len(set2conditions.get(split, []))
        n_single = sum(1 for c in set2conditions.get(split, []) if len(c.split('+')) == 2 and 'ctrl' in c)
        n_combo = sum(1 for c in set2conditions.get(split, []) if len(c.split('+')) == 2 and 'ctrl' not in c)
        print(f"{split:6s}: {n_conditions:4d} conditions  ({n_single} single-gene, {n_combo} double-gene)")

    print("\n=== Cell-level counts ===")
    for name, loader in loaders.items():
        print(f"{name:12s}: {len(loader.dataset):6d} cells")


/Users/bhumikamakwana/.pyenv/versions/3.11.4/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/bhumikamakwana/.pyenv/versions/3.11.4/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [3]:
!pip install decoupler

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.4/147.4 kB 4.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 9.3 MB/s eta 0:00:00
  Created wheel for docrep: filename=docrep-0.3.2-py3-none-any.whl size=19972 sha256=27ea6bac9e1698224679ff6581d11a8c2d5a4512e4ca872c261e49a8fdeafe26
  Stored in directory: /Users/bhumikamakwana/Library/Caches/pip/wheels/06/76/8f/0ecb7d357c0bff71a2bd1940671be2d07a200752da9189bb55
Successfully built docrep

[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import decoupler as dc

def check_collectri_overlap(pert_data):
    """Check how many genes in the Norman 2019 (GEARS) dataset are present in CollecTRI."""
    norman_genes = set(pert_data.adata.var['gene_name'].tolist())
    print(f"Norman 2019: {len(norman_genes)} genes")

    net = dc.op.collectri(organism='human', remove_complexes=False) #retrieves the CollecTRI transcriptional regulatory network for humans.
    #we're setting remove_complexes=False to include all interactions, even those involving protein complexes (more than one proteins), in the network.
    print(net.head(5))
    #net is PandasDataFrame with columns: source ──→ regulator/TF (gene or protein that initiates the regulation.), target ──→ regulated gene, weight ──→ edge/network weight, resources ──→ databases supporting the edge, references ──→ literature evidence, sign_decision ──→ how regulatory sign was determined
    collectri_genes = set(net['source']).union(set(net['target'])) #collect all genes (union of source and target genes)
    print(f"CollecTRI: {len(collectri_genes)} unique genes (TFs + targets)")
    overlap = norman_genes & collectri_genes 
    print(f"Overlap: {len(overlap)} genes ({len(overlap)/len(norman_genes):.2%} of Norman genes)")

    # count of how many CRISPR-perturbed genes specifically are covered, overlap between perturbed genes in Norman and CollecTRI
    pert_genes = set()
    for cond in pert_data.adata.obs['condition'].unique():
        for g in cond.split('+'):
            if g != 'ctrl':
                pert_genes.add(g)
    pert_overlap = pert_genes & collectri_genes 
    print(f"\nPerturbed genes in Norman: {len(pert_genes)}")
    print(f"Perturbed genes present in CollecTRI: {len(pert_overlap)} "
          f"({len(pert_overlap)/len(pert_genes):.2%})")

    collectri_tfs = set(net['source'])
    pert_tf_overlap = pert_genes & collectri_tfs
    print(f"Perturbed genes that are TFs in CollecTRI: {len(pert_tf_overlap)}")

    return {
        "norman_genes": norman_genes,
        "collectri_genes": collectri_genes,
        "overlap": overlap,
        "pert_genes": pert_genes,
        "pert_overlap": pert_overlap,
    }

In [12]:
pert_data, loaders = runner()
print_split_summary(pert_data, loaders)
overlap_results = check_collectri_overlap(pert_data)
overlap_results

Found local copy...
Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['RHOXF2BB+ctrl' 'LYL1+IER5L' 'ctrl+IER5L' 'KIAA1804+ctrl' 'IER5L+ctrl'
 'RHOXF2BB+ZBTB25' 'RHOXF2BB+SET']
Local copy of pyg dataset is detected. Loading...
Done!
Local copy of split is detected. Loading...
Simulation split test composition:
combo_seen0:9
combo_seen1:43
combo_seen2:19
unseen_single:36
Done!


here1
=== Condition-level counts (unique perturbations) ===
train :  139 conditions  (99 single-gene, 39 double-gene)
val   :   31 conditions  (13 single-gene, 18 double-gene)
test  :  107 conditions  (36 single-gene, 71 double-gene)

=== Cell-level counts ===
train_loader:  49849 cells
val_loader  :  10754 cells
test_loader :  28754 cells
Norman 2019: 5045 genes
   source target  weight                                          resources  \
0     MYC   TERT     1.0  DoRothEA-A;ExTRI;HTRI;NTNU.Curated;Pavlidis202...   
1    SPI1  BGLAP     1.0                                              ExTRI   
2   SMAD3    JUN     1.0                   ExTRI;NTNU.Curated;TFactS;TRRUST   
3   SMAD4    JUN     1.0                   ExTRI;NTNU.Curated;TFactS;TRRUST   
4  STAT5A    IL2     1.0                                              ExTRI   

                                          references       sign_decision  
0  10022128;10491298;10606235;10637317;10723141;1...                PMID  
1        

{'norman_genes': {'BARX1',
  'PJA2',
  'CELA1',
  'GADD45B',
  'KCNN2',
  'CFP',
  'BLNK',
  'RNASEL',
  'IFITM3',
  'RP11-771K4.1',
  'CORO1A',
  'CA11',
  'P2RX4',
  'SLC7A8',
  'TNFRSF14',
  'LY96',
  'MYH6',
  'RP11-266J6.2',
  'ZFHX3',
  'RP11-18H21.1',
  'TLCD2',
  'IGF1',
  'HRASLS2',
  'LGALS1',
  'RP11-44F21.5',
  'RP5-1112D6.8',
  'AC022007.5',
  'LL22NC03-104C7.1',
  'RP11-9M16.2',
  'HOTAIRM1',
  'RP11-525J21.1',
  'MYOZ3',
  'C20orf202',
  'APOL3',
  'KCNK17',
  'PDE6B',
  'GDF15',
  'ACSL5',
  'RP11-662J14.2',
  'LPAR4',
  'PSG11',
  'RP11-572B2.1',
  'IGFBP2',
  'DQX1',
  'EPPK1',
  'LINC01348',
  'CALCR',
  'LINC00327',
  'AC114803.3',
  'CLSTN2',
  'LINC00635',
  'LSAMP-AS1',
  'PBXIP1',
  'PSMB8-AS1',
  'AC007950.1',
  'STX16-NPEPL1',
  'PPP1R14D',
  'SATL1',
  'TCEAL5',
  'ENTPD8',
  'RASGRF2',
  'CHRNE',
  'FAM19A2',
  'LINC00871',
  'RP11-395E19.2',
  'SH3BGRL',
  'RP11-212I21.4',
  'RP11-818O24.3',
  'ABLIM1',
  'UCN2',
  'RP13-870H17.3',
  'BIN2',
  'ISX',
  'CAC

**CollecTRI:** 6939 unique genes (TFs + targets)

**Overlap:** **1518 genes (30.09% of Norman genes)

**Perturbed genes in Norman:** 102

**Perturbed genes present in CollecTRI:** 77 (75.49%)

**Perturbed genes that are TFs in CollecTRI:** 43

-------

Note - 
> **CollecTRI** is a targeted **TF-target regulatory network** — it doesn't aim to cover the whole transcriptome, only genes that participate in curated TF regulatory relationships (~1,186 TFs and their direct targets). 

> Norman's 5,045 genes are the top highly-variable genes from a genome-wide Perturb-seq screen, so most of them are ordinary expressed genes with no curated TF/target role

-------------------------------

> **Mechanistically-informed model structures**
To inject structural inductive bias into transcriptomics perturbation models (constraining the hypothesis space using established biological architectures).

**Integrating these three biological layers—**

[a.] **transcriptional control (Who controls whom?** -  The effect of a perturbation cascades from a TF to its target genes (regulons).)

[b.] **intracellular signaling (How does code propagate?** - control layer) Without this layer, a model is forced to map an upstream perturbation (like a drug or receptor knockout) directly to a downstream transcriptomic change, treating the entire cytoplasm as a black box.

[c.] **physical complexes (Who works together?)** proteins form physical complexes (e.g., the ribosome, proteasome, or SWI/SNF complex) or operate within tight metabolic pathways. Knocking out one member of a stable complex often destabilizes or phenocopies the loss of the entire complex.


**The Unified Graph Schema [distinct functional layer]**

Layer 3: Signaling/Kinase -> OmniPath

Layer 2: TRanscriptional control -> Transcriptional (CollecTRI)

Layer 1: Functional layers/ physical complexes -> Complexes (STRING Exp/DB)

In [2]:
!pip install omnipath

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 5.3 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


             TRANSCRIPTION
        TF ────────────────→ Gene
        │
        │
        ↓
     Protein
        │
        │ SIGNALING / PTM
        ↓
     Protein
        ↕
        │ PHYSICAL INTERACTION
     Protein

| Layer               | Database  | What it tells you                                 | Example       |
| ------------------- | --------- | ------------------------------------------------- | ------------- |
| **Transcriptional** | CollecTRI | Who regulates whose expression                    | `TP53 → BAX`  |
| **Signaling**       | OmniPath  | Who activates/inhibits whom through signaling/PTM | `AKT1 → MTOR` |
| **Physical PPI**    | STRING    | Which proteins physically/functionally interact   | `AKT1 ↔ MTOR` |


------

In [62]:
import networkx as nx
import pandas as pd
import decoupler as dc
import omnipath as op

# ---------------------------------------------------------
# 1. CollecTRI (TF-Target Regulatory Network)
# ---------------------------------------------------------
print("Fetching CollecTRI...")
collectri_df = dc.op.collectri(organism='human', remove_complexes=False)

G_collectri = nx.DiGraph()
for _, row in collectri_df.iterrows():
    G_collectri.add_edge(
        row['source'], 
        row['target'], 
        weight=row['weight'], 
        database='CollecTRI',
        interaction_type='transcriptional_regulation'
    )

print("Fetching OmniPath...")
omnipath_df = op.interactions.OmniPath().get()

# OmniPath returns UniProt IDs. 
try:
    uniprot_to_sym = op.dict.uniprot_to_symbol()
except AttributeError:
    uniprot_to_sym = {}

G_omnipath = nx.DiGraph()
for _, row in omnipath_df.iterrows():
    u_src = row.get('source')
    u_tgt = row.get('target')
    
    # Translate UniProt to Gene Symbol if dictionary exists, otherwise keep raw
    src = uniprot_to_sym.get(u_src, u_src)
    tgt = uniprot_to_sym.get(u_tgt, u_tgt)
    
    if pd.isna(src) or pd.isna(tgt):
        continue
        
    G_omnipath.add_edge(
        src, 
        tgt, 
        is_directed=row.get('is_directed', True),
        is_stimulation=row.get('is_stimulation', None),
        is_inhibition=row.get('is_inhibition', None),
        database='OmniPath',
        interaction_type='signaling'
    )


print("Downloading and parsing full STRING network for human (9606)...")

info_url = "https://stringdb-static.org/download/protein.info.v12.0/9606.protein.info.v12.0.txt.gz"
links_url = "https://stringdb-static.org/download/protein.links.v12.0/9606.protein.links.v12.0.txt.gz"

protein_info = pd.read_csv(info_url, sep='\t', compression='gzip')
id_to_symbol = dict(zip(protein_info['#string_protein_id'], protein_info['preferred_name']))

links_df = pd.read_csv(links_url, sep=' ', compression='gzip')
links_df = links_df[links_df['combined_score'] >= 400] 

G_string = nx.DiGraph()
for _, row in links_df.iterrows():
    p1 = id_to_symbol.get(row['protein1'])
    p2 = id_to_symbol.get(row['protein2'])
    
    if p1 and p2:
        G_string.add_edge(
            p1, p2, 
            score=row['combined_score'], 
            database='STRING',
            interaction_type='functional_association',
            is_directional=False
        )
        
print("Merging graphs into a MultiDiGraph...")
G_merged = nx.MultiDiGraph()

def merge_into_multigraph(target_G, source_G):
    for u, v, edge_data in source_G.edges(data=True):
        target_G.add_edge(u, v, **edge_data)

merge_into_multigraph(G_merged, G_collectri)
merge_into_multigraph(G_merged, G_omnipath)
merge_into_multigraph(G_merged, G_string)

print(f"\n--- Merged Network Summary ---")
print(f"Total Nodes: {G_merged.number_of_nodes():,}")
print(f"Total Edges: {G_merged.number_of_edges():,}")

Fetching CollecTRI...
Fetching OmniPath...
Merging graphs into a MultiDiGraph...

--- Merged Network Summary ---
Total Nodes: 28,514
Total Edges: 1,988,123


**Retrieving all interactions between two specific genes**

In [63]:
source_gene = "TP53"
target_gene = "MYC"

if G_merged.has_edge(source_gene, target_gene):
    # This returns a dictionary of dictionaries containing all parallel edges
    edges = G_merged[source_gene][target_gene]
    
    print(f"Found {len(edges)} interactions from {source_gene} to {target_gene}:")
    
    for edge_idx, edge_attr in edges.items():
        db = edge_attr.get('database')
        int_type = edge_attr.get('interaction_type')
        
        print(f"\n--- Edge {edge_idx} ({db}) ---")
        print(f"Type: {int_type}")
        
        # Database-specific attributes
        if db == 'CollecTRI':
            effect = "Activation" if edge_attr.get('weight') > 0 else "Inhibition"
            print(f"Effect: {effect}")
        elif db == 'OmniPath':
            print(f"Is Stimulation: {edge_attr.get('is_stimulation')}")
            print(f"Is Inhibition: {edge_attr.get('is_inhibition')}")
        elif db == 'STRING':
            print(f"Confidence Score: {edge_attr.get('score')}")
else:
    print("No interactions found between these genes.")

Found 2 interactions from TP53 to MYC:

--- Edge 0 (CollecTRI) ---
Type: transcriptional_regulation
Effect: Inhibition

--- Edge 1 (STRING) ---
Type: functional_association
Confidence Score: 997


**Finding all downstream targets of a specific gene**

In [64]:
gene = "EGFR"

if gene in G_merged:
    targets = list(G_merged.successors(gene))
    print(f"{gene} points to {len(targets)} unique genes.")
    
    # Example: Look at the first 5 targets
    for target in targets[:5]:
        interactions = G_merged[gene][target]
        dbs = [attr['database'] for _, attr in interactions.items()]
        print(f" -> {target} (Found in: {', '.join(dbs)})")

EGFR points to 1527 unique genes.
 -> AKT1 (Found in: STRING)
 -> RARB (Found in: STRING)
 -> CNKSR2 (Found in: STRING)
 -> VIRMA (Found in: STRING)
 -> SERPINB5 (Found in: STRING)


**Get only ultra-high confidence STRING interactions (>900)**

In [65]:
high_conf_string = [
    (u, v, attr) for u, v, attr in G_merged.edges(data=True) 
    if attr.get('database') == 'STRING' and attr.get('score', 0) >= 900
]
print(f"Found {len(high_conf_string)} ultra-high confidence physical interactions.")

Found 201712 ultra-high confidence physical interactions.
